# 🌊 Building Research Software the Right Way
## A Gravitational-Wave Data Analysis Package — from Zero to Installable

This notebook walks you through creating a **real, installable Python package** for gravitational-wave (GW) signal processing, following all 10 guidelines from:

> *"Ten Essential Guidelines for Building High-Quality Research Software"*  
> Eisty et al., arXiv:2507.16166v1 (2025)

### What we build
`gwstrain` — a small package that:
- Generates synthetic GW strain data (noise + chirp signal)
- Saves/loads data in the community-standard HDF5 format
- Applies matched-filter signal processing to detect the signal
- Can be installed by anyone with `pip install .` or `uv add gwstrain`

### Guideline coverage at a glance
| # | Guideline | Where shown |
|---|-----------|-------------|
| 1 | Plan Before You Code | §1 — design decisions written out |
| 2 | Design for Modularity | §2 — three independent modules |
| 3 | Write Clean & Readable Code | §3 — linting, naming, single-purpose functions |
| 4 | Use Version Control | §4 — `git init`, commits, `.gitignore` |
| 5 | Test Regularly | §5 — `pytest` unit + integration tests |
| 6 | Peer Code Review | §6 — PR checklist template |
| 7 | Document Everything | §7 — docstrings, README, Sphinx |
| 8 | Reproducibility | §8 — seeds, HDF5 metadata, environment lock |
| 9 | Performance & Scalability | §9 — profiling, FFT optimisation |
| 10 | Long-Term Maintenance | §10 — CI workflow, version pinning |

---
## ① Guideline 1 — Plan Before You Code

Before a single line of Python, we write down our **requirements** and **architecture**.

### Problem statement
Researchers need to:
1. Generate reproducible synthetic GW strain time-series (detector noise + compact-binary chirp signal)
2. Persist that data in a shareable format (HDF5) with full metadata
3. Run a matched-filter pipeline to recover the signal from noise

### Scope (MVP vs nice-to-have)
**In scope:** noise generation, chirp injection, HDF5 I/O, matched filter, CLI entry-point  
**Out of scope (v0):** real LIGO data frames, PSD estimation, parameter estimation

### Module architecture
```
gwstrain/
├── __init__.py          ← public API
├── noise.py             ← Gaussian / colored noise generation   (Module A)
├── waveform.py          ← chirp signal template generation       (Module B)
├── io.py                ← HDF5 save / load with metadata         (Module C)
└── matched_filter.py    ← signal processing pipeline             (Module D)
```

Each module has **one responsibility** → see Guideline 2.

### Risk: numerical reproducibility across platforms
We will always accept a `seed` parameter and store it in file metadata (Guideline 8).

---
## ④ Guideline 4 — Version Control & Project Setup with `uv`

We set up the project with `uv` (the fast modern Python package manager) and immediately commit to Git.

> **Run the shell cells below in a terminal** (or as notebook cells if your kernel has shell access).  
> The `%%bash` magic works in Jupyter when running locally.

```bash
# ── 1. Create the project with uv ─────────────────────────────────────────
uv init gwstrain          # scaffolds pyproject.toml, src layout, .python-version
cd gwstrain

# ── 2. Add runtime dependencies ───────────────────────────────────────────
uv add numpy scipy h5py matplotlib

# ── 3. Add dev-only dependencies ──────────────────────────────────────────
uv add --dev pytest pytest-cov flake8

# ── 4. Initialise Git and make the first commit ───────────────────────────
git init
git add .
git commit -m "chore: initialise project with uv"

# ── 5. Connect to GitHub (replace with your repo URL) ─────────────────────
git remote add origin https://github.com/YOUR_USERNAME/gwstrain.git
git push -u origin main
```

### What `uv init` creates for you
```
gwstrain/
├── pyproject.toml     ← PEP 621 package metadata + dependency pins
├── uv.lock            ← exact lock file for reproducible installs  (commit this!)
├── .python-version    ← pins the Python interpreter version
├── README.md
└── src/
    └── gwstrain/
        └── __init__.py
```

### `.gitignore` (important!)
```gitignore
# Python
__pycache__/
*.py[cod]
.venv/
dist/
*.egg-info/

# Data — don't commit large HDF5 files to Git
*.h5
*.hdf5
data/

# IDE
.vscode/
.idea/
```

### Branching strategy
We use a simple trunk-based workflow:
- `main` — always deployable
- `feat/<name>` — short-lived feature branches → merged via Pull Request
- `fix/<name>` — bug fixes


---
## ② ③ Guidelines 2 & 3 — Modular Design + Clean Code

Now we write the four modules. Notice:
- Every function does **one thing** (single-responsibility)
- All names are **descriptive** (`generate_colored_noise`, not `gen` or `f1`)
- Every public function has a **NumPy-style docstring**
- We avoid code duplication by sharing helpers across modules

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# We run everything in-notebook for demonstration.
# In the real project each block below lives in its own file:
#   src/gwstrain/noise.py, waveform.py, io.py, matched_filter.py
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import scipy.signal as signal
import scipy.fft as fft
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

print("Dependencies imported OK")

### Module A — `noise.py`  
Generate Gaussian detector noise with a simple 1/f (pink) coloring.

In [ ]:
# ── src/gwstrain/noise.py ────────────────────────────────────────────────────

def generate_white_noise(
    n_samples: int,
    sample_rate: float,
    amplitude: float = 1.0,
    seed: int | None = None,
) -> np.ndarray:
    """Generate white Gaussian noise.

    Parameters
    ----------
    n_samples : int
        Number of time-domain samples to produce.
    sample_rate : float
        Sampling rate in Hz.  Used to normalise amplitude to 1/sqrt(Hz).
    amplitude : float
        Overall noise amplitude scale factor.
    seed : int or None
        Random seed for reproducibility.  Pass the same seed to get the
        same noise realisation across runs and platforms.

    Returns
    -------
    np.ndarray, shape (n_samples,)
        Zero-mean Gaussian noise time series.
    """
    rng = np.random.default_rng(seed)  # modern, reproducible API
    noise = rng.standard_normal(n_samples)
    # Normalise to units of 1/sqrt(Hz) — standard in GW literature
    return noise * amplitude / np.sqrt(sample_rate)


def generate_colored_noise(
    n_samples: int,
    sample_rate: float,
    asd_func=None,
    amplitude: float = 1.0,
    seed: int | None = None,
) -> np.ndarray:
    """Generate noise coloured by an amplitude spectral density (ASD) function.

    Procedure:
    1. Generate white noise in the frequency domain.
    2. Multiply by the desired ASD shape.
    3. Inverse FFT back to the time domain.

    Parameters
    ----------
    n_samples : int
        Number of time-domain samples.
    sample_rate : float
        Sampling rate in Hz.
    asd_func : callable or None
        Function ``f -> ASD(f)`` that returns the desired amplitude spectral
        density at each frequency in Hz.  If None, white noise is returned.
    amplitude : float
        Overall noise amplitude scale factor.
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    np.ndarray, shape (n_samples,)
        Coloured noise time series.
    """
    rng = np.random.default_rng(seed)
    freqs = fft.rfftfreq(n_samples, d=1.0 / sample_rate)

    # White noise in frequency domain (complex)
    white_fd = rng.standard_normal(len(freqs)) + 1j * rng.standard_normal(len(freqs))

    if asd_func is not None:
        # Avoid DC singularity for 1/f-type functions
        safe_freqs = np.where(freqs > 0, freqs, freqs[1])
        white_fd *= asd_func(safe_freqs)

    colored_td = fft.irfft(white_fd, n=n_samples)
    # Normalise to unit variance
    std = colored_td.std()
    if std > 0:
        colored_td /= std
    return colored_td * amplitude / np.sqrt(sample_rate)


def simple_asd(f: np.ndarray) -> np.ndarray:
    """Toy Advanced-LIGO-inspired ASD: flat above 50 Hz, rising below."""
    asd = np.ones_like(f)
    low_freq_mask = f < 50.0
    asd[low_freq_mask] = (50.0 / f[low_freq_mask]) ** 2  # 1/f^2 below 50 Hz
    return asd


print("Module A (noise.py) defined")

### Module B — `waveform.py`  
Generate a simple chirp signal that mimics a compact binary coalescence.

In [ ]:
# ── src/gwstrain/waveform.py ─────────────────────────────────────────────────

def chirp_signal(
    times: np.ndarray,
    f0: float = 30.0,
    f1: float = 300.0,
    amplitude: float = 1.0,
    t_coalescence: float | None = None,
) -> np.ndarray:
    """Generate a compact-binary-inspired linear chirp waveform.

    This is a pedagogical approximation — a linearly frequency-swept sine
    with a Tukey amplitude envelope — not a full post-Newtonian waveform.

    Parameters
    ----------
    times : np.ndarray
        Time array in seconds.
    f0 : float
        Starting frequency in Hz.
    f1 : float
        Ending frequency in Hz (at coalescence).
    amplitude : float
        Peak signal amplitude.
    t_coalescence : float or None
        Time of peak amplitude (coalescence). Defaults to the end of the
        time array.

    Returns
    -------
    np.ndarray, shape like ``times``
        GW strain time series h(t).
    """
    if t_coalescence is None:
        t_coalescence = times[-1]

    duration = t_coalescence - times[0]
    # Instantaneous phase for a linear chirp
    phase = 2.0 * np.pi * (f0 * (times - times[0]) +
                           0.5 * (f1 - f0) / duration * (times - times[0]) ** 2)

    # Tukey (tapered cosine) envelope: amplitude rises then stays flat, peaks at coalescence
    envelope = _tukey_envelope(times, t_coalescence, alpha=0.5)
    return amplitude * envelope * np.sin(phase)


def _tukey_envelope(times: np.ndarray, t_peak: float, alpha: float = 0.5) -> np.ndarray:
    """Build a Tukey window anchored at t_peak with taper fraction alpha.

    This is a private helper (underscore prefix) — not part of the public API.
    """
    n = len(times)
    window = signal.windows.tukey(n, alpha=alpha)
    # Shift the peak to t_peak
    peak_idx = np.argmin(np.abs(times - t_peak))
    shift = peak_idx - (n - 1)  # shift so peak aligns with right edge
    window = np.roll(window, shift)
    window[:max(0, shift)] = 0.0  # zero out wrapped-around portion
    return window


def inject_signal(
    strain: np.ndarray,
    template: np.ndarray,
    snr_target: float = 8.0,
    noise_std: float | None = None,
) -> tuple[np.ndarray, float]:
    """Inject a signal template into a strain array at a target SNR.

    Parameters
    ----------
    strain : np.ndarray
        Background noise time series to inject into.
    template : np.ndarray
        Signal template (same length as strain).
    snr_target : float
        Desired optimal matched-filter SNR.
    noise_std : float or None
        Standard deviation of the background noise.  Estimated from
        ``strain`` if not provided.

    Returns
    -------
    injected : np.ndarray
        Strain + scaled signal.
    scale : float
        Scale factor applied to the template.
    """
    if noise_std is None:
        noise_std = strain.std()

    # Optimal SNR of a template matched to itself: SNR = ||h|| / sigma_noise
    template_norm = np.sqrt(np.sum(template ** 2))
    if template_norm == 0:
        raise ValueError("Template has zero norm — cannot inject signal.")

    scale = snr_target * noise_std / template_norm
    return strain + scale * template, scale


print("Module B (waveform.py) defined")

### Module C — `io.py`  
HDF5 save/load with full reproducibility metadata (Guideline 8).

In [ ]:
# ── src/gwstrain/io.py ───────────────────────────────────────────────────────
import datetime

GWSTRAIN_FORMAT_VERSION = "1.0"


def save_strain(
    path: str | Path,
    times: np.ndarray,
    strain: np.ndarray,
    sample_rate: float,
    metadata: dict | None = None,
) -> None:
    """Save a GW strain time series to an HDF5 file.

    The file structure follows the community-recommended layout::

        /
        ├── strain          float64 array, length N
        ├── times           float64 array, length N
        └── attrs:
              sample_rate          float
              gwstrain_version     str   (format version for forwards-compat)
              created_utc          str   (ISO 8601 timestamp)
              + any user-supplied metadata

    Parameters
    ----------
    path : str or Path
        Output file path.  Will be created or overwritten.
    times : np.ndarray
        Time array in seconds.
    strain : np.ndarray
        Strain data (same length as times).
    sample_rate : float
        Sampling rate in Hz.
    metadata : dict or None
        Arbitrary key/value pairs stored as HDF5 root attributes.  Values
        must be HDF5-serialisable (strings, ints, floats, arrays).
    """
    if len(times) != len(strain):
        raise ValueError(
            f"times and strain must have equal length: {len(times)} != {len(strain)}"
        )

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with h5py.File(path, "w") as f:
        f.create_dataset("strain", data=strain, compression="gzip", compression_opts=4)
        f.create_dataset("times",  data=times,  compression="gzip", compression_opts=4)

        # Always-present provenance metadata
        f.attrs["sample_rate"]        = float(sample_rate)
        f.attrs["gwstrain_version"]   = GWSTRAIN_FORMAT_VERSION
        f.attrs["created_utc"]        = datetime.datetime.utcnow().isoformat()

        # User-supplied metadata
        if metadata:
            for key, value in metadata.items():
                f.attrs[key] = value

    print(f"Saved {len(strain)} samples → {path}")


def load_strain(path: str | Path) -> dict:
    """Load a GW strain file saved by :func:`save_strain`.

    Returns
    -------
    dict with keys:
        - ``times``       np.ndarray
        - ``strain``      np.ndarray
        - ``sample_rate`` float
        - ``metadata``    dict  (all HDF5 root attributes)
    """
    with h5py.File(path, "r") as f:
        times       = f["times"][:]
        strain      = f["strain"][:]
        sample_rate = float(f.attrs["sample_rate"])
        metadata    = dict(f.attrs)

    return {
        "times":       times,
        "strain":      strain,
        "sample_rate": sample_rate,
        "metadata":    metadata,
    }


print("Module C (io.py) defined")

### Module D — `matched_filter.py`  
Signal processing: bandpass filter + matched filter SNR time series.

In [ ]:
# ── src/gwstrain/matched_filter.py ──────────────────────────────────────────

def bandpass(
    strain: np.ndarray,
    sample_rate: float,
    low_hz: float = 30.0,
    high_hz: float = 350.0,
    order: int = 4,
) -> np.ndarray:
    """Apply a zero-phase Butterworth bandpass filter to strain data.

    Zero-phase filtering (``sosfiltfilt``) avoids group-delay distortion,
    which is critical for GW transient searches.

    Parameters
    ----------
    strain : np.ndarray
        Input strain time series.
    sample_rate : float
        Sampling rate in Hz.
    low_hz : float
        Lower passband edge in Hz.
    high_hz : float
        Upper passband edge in Hz.
    order : int
        Filter order.

    Returns
    -------
    np.ndarray
        Bandpass-filtered strain, same length as input.
    """
    nyquist = sample_rate / 2.0
    sos = signal.butter(
        order,
        [low_hz / nyquist, high_hz / nyquist],
        btype="band",
        output="sos",
    )
    return signal.sosfiltfilt(sos, strain)


def matched_filter_snr(
    strain: np.ndarray,
    template: np.ndarray,
    sample_rate: float,
    psd: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute the matched-filter SNR time series.

    The matched filter is optimal for detecting a known signal in
    stationary Gaussian noise.  In the frequency domain::

        SNR(t) = IFFT[ FFT(h) * conj(FFT(template)) / PSD(f) ]

    Parameters
    ----------
    strain : np.ndarray
        Detector strain time series (signal + noise).
    template : np.ndarray
        Expected signal template (same length as strain).
    sample_rate : float
        Sampling rate in Hz.
    psd : np.ndarray or None
        One-sided PSD evaluated at ``rfftfreq(len(strain), 1/sample_rate)``.
        If None, unit PSD is assumed (whitened data).

    Returns
    -------
    times : np.ndarray
        Time array corresponding to the SNR time series.
    snr : np.ndarray
        Absolute matched-filter SNR at each time sample.
    """
    n = len(strain)
    dt = 1.0 / sample_rate
    times = np.arange(n) * dt

    strain_fd   = fft.rfft(strain)
    template_fd = fft.rfft(template)
    freqs       = fft.rfftfreq(n, d=dt)

    if psd is None:
        psd = np.ones_like(freqs)

    # Avoid dividing by zero at DC
    safe_psd = np.where(psd > 0, psd, 1.0)

    # Frequency-domain matched filter
    integrand = strain_fd * np.conj(template_fd) / safe_psd
    snr_complex = fft.irfft(integrand, n=n)

    # Normalise by the template norm
    sigma = np.sqrt(np.sum(np.abs(template_fd) ** 2 / safe_psd) / n)
    if sigma > 0:
        snr_complex /= sigma

    return times, np.abs(snr_complex)


def find_peak(
    times: np.ndarray,
    snr: np.ndarray,
    threshold: float = 5.0,
) -> dict | None:
    """Find the loudest SNR peak above a detection threshold.

    Parameters
    ----------
    times : np.ndarray
        Time array.
    snr : np.ndarray
        SNR time series.
    threshold : float
        Detection threshold SNR value.

    Returns
    -------
    dict or None
        ``{'time': float, 'snr': float}`` for the loudest peak, or
        ``None`` if no samples exceed the threshold.
    """
    above_threshold = snr > threshold
    if not np.any(above_threshold):
        return None
    peak_idx = np.argmax(snr)
    return {"time": float(times[peak_idx]), "snr": float(snr[peak_idx])}


print("Module D (matched_filter.py) defined")

---
## ⑧ Guideline 8 — Reproducibility: Generate & Save Data

In [ ]:
# ── Reproducibility parameters — always store these! ────────────────────────
RANDOM_SEED    = 42       # <- change this to explore different noise realisations
SAMPLE_RATE    = 4096.0   # Hz  (standard LIGO low-latency rate)
DURATION_SEC   = 4.0      # seconds of data
CHIRP_SNR      = 10.0     # target matched-filter SNR
COALESCENCE_T  = 3.5      # seconds into the segment

n_samples = int(DURATION_SEC * SAMPLE_RATE)
times = np.arange(n_samples) / SAMPLE_RATE

# ── Generate coloured noise (Module A) ─────────────────────────────────────
noise = generate_colored_noise(
    n_samples=n_samples,
    sample_rate=SAMPLE_RATE,
    asd_func=simple_asd,
    amplitude=1.0,
    seed=RANDOM_SEED,
)

# ── Generate chirp template (Module B) ────────────────────────────────────
template = chirp_signal(
    times=times,
    f0=30.0,
    f1=300.0,
    amplitude=1.0,
    t_coalescence=COALESCENCE_T,
)

# ── Inject signal into noise (Module B) ───────────────────────────────────
strain, injection_scale = inject_signal(
    strain=noise.copy(),
    template=template,
    snr_target=CHIRP_SNR,
)

print(f"Generated {n_samples:,} samples at {SAMPLE_RATE} Hz")
print(f"Signal injected with scale factor: {injection_scale:.4e}")
print(f"Strain std (noise+signal): {strain.std():.4e}")

In [ ]:
# ── Save to HDF5 with full metadata (Module C) ───────────────────────────────
output_path = Path("data/simulated_strain.h5")

save_strain(
    path=output_path,
    times=times,
    strain=strain,
    sample_rate=SAMPLE_RATE,
    metadata={
        # Guideline 8: record every parameter needed to reproduce this file
        "random_seed":     RANDOM_SEED,
        "duration_sec":    DURATION_SEC,
        "chirp_snr":       CHIRP_SNR,
        "coalescence_t":   COALESCENCE_T,
        "chirp_f0_hz":     30.0,
        "chirp_f1_hz":     300.0,
        "noise_model":     "colored_gaussian_simple_asd",
        "injection_scale": injection_scale,
        "generator":       "gwstrain v0.1.0",
    },
)

# ── Reload and verify round-trip integrity ────────────────────────────────
loaded = load_strain(output_path)
assert np.allclose(loaded["strain"], strain), "Round-trip mismatch!"
print("\nMetadata stored in file:")
for k, v in loaded["metadata"].items():
    print(f"  {k:25s} = {v}")

---
## Signal Processing: Bandpass + Matched Filter

Now we load the saved file (as a downstream user would) and run the pipeline.

In [ ]:
# ── Load data just as a user would ───────────────────────────────────────────
data = load_strain(output_path)
t       = data["times"]
h       = data["strain"]
fs      = data["sample_rate"]

# ── Step 1: Bandpass filter (Module D) ───────────────────────────────────────
h_bp = bandpass(h, sample_rate=fs, low_hz=30.0, high_hz=350.0)

# ── Step 2: Matched filter (Module D) ────────────────────────────────────────
# Re-create the same template (in production this would come from a template bank)
tmpl = chirp_signal(t, f0=30.0, f1=300.0, t_coalescence=COALESCENCE_T)

snr_times, snr = matched_filter_snr(
    strain=h_bp,
    template=tmpl,
    sample_rate=fs,
)

# ── Step 3: Trigger detection ─────────────────────────────────────────────────
trigger = find_peak(snr_times, snr, threshold=5.0)
if trigger:
    print(f"\n🎯 Detection trigger!")
    print(f"   Peak SNR  : {trigger['snr']:.2f}")
    print(f"   Peak time : {trigger['time']:.3f} s  (injected at {COALESCENCE_T} s)")
else:
    print("No trigger above threshold.")

In [ ]:
# ── Diagnostic plot ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
fig.suptitle("gwstrain — Simulated GW Detection Pipeline", fontsize=14, fontweight="bold")

# Panel 1: Raw strain
axes[0].plot(t, h, lw=0.5, color="steelblue", alpha=0.8, label="Strain (noise + signal)")
axes[0].axvline(COALESCENCE_T, color="crimson", ls="--", lw=1.5, label="Injected coalescence")
axes[0].set_ylabel("Strain [arb.]")
axes[0].set_title("Raw strain")
axes[0].legend(fontsize=9)

# Panel 2: Bandpassed + template overlay
axes[1].plot(t, h_bp, lw=0.6, color="darkorange", alpha=0.9, label="Bandpass-filtered")
axes[1].plot(t, tmpl * injection_scale, lw=1.5, color="crimson", label="True signal (scaled)")
axes[1].set_ylabel("Strain [arb.]")
axes[1].set_title("Bandpass-filtered strain + injected signal")
axes[1].legend(fontsize=9)

# Panel 3: Matched-filter SNR
axes[2].plot(snr_times, snr, lw=1.0, color="seagreen", label="MF-SNR")
axes[2].axhline(5.0, color="gray", ls=":", lw=1.5, label="Threshold SNR=5")
if trigger:
    axes[2].axvline(trigger["time"], color="crimson", ls="--", lw=1.5,
                    label=f"Peak SNR={trigger['snr']:.1f} @ t={trigger['time']:.3f}s")
axes[2].set_xlabel("Time [s]")
axes[2].set_ylabel("SNR")
axes[2].set_title("Matched-filter SNR")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig("data/gw_pipeline_result.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

---
## ⑤ Guideline 5 — Test Your Code Regularly

Real tests live in `tests/`.  Here are the complete test files.

In [ ]:
# ── tests/test_noise.py ──────────────────────────────────────────────────────
TEST_NOISE_PY = '''
"""Unit tests for gwstrain.noise."""
import numpy as np
import pytest
from gwstrain.noise import generate_white_noise, generate_colored_noise, simple_asd


def test_white_noise_shape():
    """Output length must match n_samples."""
    noise = generate_white_noise(n_samples=1024, sample_rate=4096.0, seed=0)
    assert noise.shape == (1024,)


def test_white_noise_reproducibility():
    """Same seed must produce identical noise (Guideline 8)."""
    n1 = generate_white_noise(512, 4096.0, seed=99)
    n2 = generate_white_noise(512, 4096.0, seed=99)
    np.testing.assert_array_equal(n1, n2)


def test_white_noise_different_seeds():
    """Different seeds must produce different noise."""
    n1 = generate_white_noise(512, 4096.0, seed=1)
    n2 = generate_white_noise(512, 4096.0, seed=2)
    assert not np.array_equal(n1, n2)


def test_white_noise_zero_mean():
    """White noise must be approximately zero-mean for large N."""
    noise = generate_white_noise(n_samples=100_000, sample_rate=4096.0, seed=42)
    assert abs(noise.mean()) < 0.01


def test_colored_noise_shape():
    noise = generate_colored_noise(1024, 4096.0, asd_func=simple_asd, seed=0)
    assert noise.shape == (1024,)


def test_simple_asd_flat_above_50hz():
    """simple_asd should return 1.0 at 100 Hz."""
    freqs = np.array([100.0, 200.0, 500.0])
    assert np.allclose(simple_asd(freqs), 1.0)
'''

# ── tests/test_io.py ─────────────────────────────────────────────────────────
TEST_IO_PY = '''
"""Integration tests for gwstrain.io — covers HDF5 round-trip."""
import numpy as np
import pytest
from pathlib import Path
from gwstrain.io import save_strain, load_strain


def test_save_and_load_roundtrip(tmp_path):
    """Data written by save_strain must be recovered exactly by load_strain."""
    rng = np.random.default_rng(0)
    n   = 1000
    t   = np.linspace(0, 1, n)
    h   = rng.standard_normal(n)

    path = tmp_path / "test.h5"
    save_strain(path, t, h, sample_rate=1000.0, metadata={"seed": 0})
    loaded = load_strain(path)

    np.testing.assert_allclose(loaded["times"],  t,    rtol=1e-12)
    np.testing.assert_allclose(loaded["strain"], h,    rtol=1e-12)
    assert loaded["sample_rate"] == 1000.0
    assert loaded["metadata"]["seed"] == 0


def test_mismatched_lengths_raises(tmp_path):
    """save_strain must raise ValueError when times and strain differ in length."""
    with pytest.raises(ValueError, match="equal length"):
        save_strain(tmp_path / "bad.h5",
                    times=np.zeros(10), strain=np.zeros(11), sample_rate=1.0)


def test_metadata_stored(tmp_path):
    path = tmp_path / "meta.h5"
    save_strain(path, np.zeros(5), np.zeros(5), sample_rate=100.0,
                metadata={"experiment": "gw_sim", "snr": 8.0})
    loaded = load_strain(path)
    assert loaded["metadata"]["experiment"] == "gw_sim"
    assert loaded["metadata"]["snr"] == pytest.approx(8.0)
'''

# Write test files to disk
import os
os.makedirs("tests", exist_ok=True)
Path("tests/test_noise.py").write_text(TEST_NOISE_PY)
Path("tests/test_io.py").write_text(TEST_IO_PY)
print("Test files written to tests/")

---
## ⑦ Guideline 7 — Documentation

### `pyproject.toml` — the single source of truth

In [ ]:
PYPROJECT_TOML = '''
[build-system]
requires      = ["hatchling"]
build-backend = "hatchling.build"

[project]
name        = "gwstrain"
version     = "0.1.0"
description = "Synthetic gravitational-wave strain generation and matched-filter detection"
readme      = "README.md"
license     = { text = "MIT" }
requires-python = ">=3.11"

dependencies = [
    "numpy>=1.26",
    "scipy>=1.12",
    "h5py>=3.10",
    "matplotlib>=3.8",
]

[project.optional-dependencies]
dev = [
    "pytest>=8.0",
    "pytest-cov",
    "flake8",
]

[project.scripts]
# CLI entry-point: users can run  `gwstrain-demo`  after install
gwstrain-demo = "gwstrain.cli:main"

[project.urls]
Homepage   = "https://github.com/YOUR_USERNAME/gwstrain"
Issues     = "https://github.com/YOUR_USERNAME/gwstrain/issues"

[tool.pytest.ini_options]
addopts     = "--tb=short -q --cov=gwstrain --cov-report=term-missing"
testpaths   = ["tests"]

[tool.flake8]
max-line-length = 100
extend-ignore   = ["E203", "W503"]
'''
print(PYPROJECT_TOML)

In [ ]:
README_MD = '''
# gwstrain

[![CI](https://github.com/YOUR_USERNAME/gwstrain/actions/workflows/ci.yml/badge.svg)](https://github.com/YOUR_USERNAME/gwstrain/actions)
[![License: MIT](https://img.shields.io/badge/License-MIT-blue.svg)](LICENSE)

A small, pedagogical Python package for **synthetic gravitational-wave strain generation
and matched-filter signal detection**.

## Installation

```bash
# With uv (recommended)
uv add gwstrain

# Or with pip
pip install gwstrain

# Developer install (editable)
git clone https://github.com/YOUR_USERNAME/gwstrain.git
cd gwstrain
uv sync --dev    # installs all deps + dev extras into .venv
```

## Quick Start

```python
import numpy as np
from gwstrain.noise import generate_colored_noise, simple_asd
from gwstrain.waveform import chirp_signal, inject_signal
from gwstrain.io import save_strain, load_strain
from gwstrain.matched_filter import bandpass, matched_filter_snr, find_peak

fs, T, seed = 4096.0, 4.0, 42
t = np.arange(int(T * fs)) / fs

noise    = generate_colored_noise(len(t), fs, simple_asd, seed=seed)
template = chirp_signal(t, f0=30, f1=300, t_coalescence=3.5)
strain, _ = inject_signal(noise, template, snr_target=10)

save_strain("data/strain.h5", t, strain, fs, metadata={"seed": seed})

data             = load_strain("data/strain.h5")
h_bp             = bandpass(data["strain"], fs)
snr_t, snr       = matched_filter_snr(h_bp, template, fs)
trigger          = find_peak(snr_t, snr)
print(trigger)   # {'time': 3.50x, 'snr': ~10.x}
```

## Running tests

```bash
uv run pytest
```

## Contributing

See [CONTRIBUTING.md](CONTRIBUTING.md).  Good first issues are tagged `good-first-issue`.

## Citation

If you use gwstrain in a publication, please cite the companion guidelines paper:

> Eisty et al. (2025). "Ten Essential Guidelines for Building High-Quality Research Software."  
> arXiv:2507.16166
'''
print(README_MD)

---
## ④ Guideline 4 (cont.) — Git Commit Workflow

After writing each module, commit with **atomic, descriptive messages**:

In [ ]:
GIT_WORKFLOW = """
# After initial uv init + git init:
git commit -m "chore: initialise project with uv"

# Add .gitignore
git add .gitignore
git commit -m "chore: add .gitignore for Python, data files, and IDE"

# Module by module — one commit per logical unit
git add src/gwstrain/noise.py
git commit -m "feat(noise): add white and colored noise generators with seed support"

git add src/gwstrain/waveform.py
git commit -m "feat(waveform): add chirp signal and inject_signal with target-SNR scaling"

git add src/gwstrain/io.py
git commit -m "feat(io): add HDF5 save/load with provenance metadata"

git add src/gwstrain/matched_filter.py
git commit -m "feat(matched_filter): add bandpass, matched_filter_snr, find_peak"

# Tests
git add tests/
git commit -m "test: add unit and integration tests for noise and io modules"

# Docs
git add README.md CONTRIBUTING.md pyproject.toml
git commit -m "docs: add README with quickstart, installation, and citation"

# CI
git add .github/
git commit -m "ci: add GitHub Actions workflow to run pytest on push/PR"

# Push
git push origin main
"""
print(GIT_WORKFLOW)

---
## ⑩ Guideline 10 — Long-Term Maintenance: CI/CD with GitHub Actions

Save this as `.github/workflows/ci.yml` in your repo:

In [ ]:
CI_YML = '''
# .github/workflows/ci.yml
name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11", "3.12"]   # test across Python versions

    steps:
      - uses: actions/checkout@v4

      - name: Install uv
        uses: astral-sh/setup-uv@v3
        with:
          version: "latest"

      - name: Set up Python ${{ matrix.python-version }}
        run: uv python install ${{ matrix.python-version }}

      - name: Install dependencies
        run: uv sync --dev

      - name: Lint with flake8
        run: uv run flake8 src/gwstrain

      - name: Run tests with coverage
        run: uv run pytest --cov=gwstrain --cov-report=xml

      - name: Upload coverage to Codecov
        uses: codecov/codecov-action@v4
        with:
          files: coverage.xml
'''
print(CI_YML)

---
## ⑥ Guideline 6 — Peer Code Review: PR Template

Save as `.github/PULL_REQUEST_TEMPLATE.md`:

In [ ]:
PR_TEMPLATE = '''
## Summary
<!-- What does this PR change and why? -->

## Type of change
- [ ] Bug fix
- [ ] New feature
- [ ] Refactor / clean-up
- [ ] Documentation
- [ ] CI / tooling

## Guideline checklist (Eisty et al. 2025)
- [ ] **Clean code**: functions have single responsibility, descriptive names
- [ ] **Tests**: new code has unit/integration tests; all existing tests pass
- [ ] **Documentation**: docstrings updated; README updated if API changed
- [ ] **Reproducibility**: any new parameters exposed as arguments with defaults
- [ ] **No data files committed** (use data/ which is .gitignored)

## Testing done
```
uv run pytest           # paste output here
uv run flake8 src/
```
'''
print(PR_TEMPLATE)

---
## ⑨ Guideline 9 — Performance: Profile Before Optimising

In [ ]:
import cProfile, pstats, io

# ── Profile the full pipeline ─────────────────────────────────────────────────
def run_pipeline():
    """End-to-end pipeline call — what we actually profile."""
    n = int(4.0 * 4096)
    t = np.arange(n) / 4096.0
    noise_    = generate_colored_noise(n, 4096.0, simple_asd, seed=0)
    tmpl_     = chirp_signal(t, 30, 300, t_coalescence=3.5)
    strain_, _ = inject_signal(noise_, tmpl_, snr_target=10.0)
    h_bp_     = bandpass(strain_, 4096.0)
    _, snr_   = matched_filter_snr(h_bp_, tmpl_, 4096.0)
    return snr_.max()

pr = cProfile.Profile()
pr.enable()
peak = run_pipeline()
pr.disable()

stream = io.StringIO()
ps = pstats.Stats(pr, stream=stream).sort_stats("cumulative")
ps.print_stats(10)   # top 10 slowest calls
print(stream.getvalue())
print(f"Peak SNR from profiled run: {peak:.2f}")

---
## 🎁 How Others Install and Use Your Package

Once you've pushed to GitHub, anyone can install with:

```bash
# From PyPI (after uv publish or twine upload)
pip install gwstrain
uv add gwstrain

# Directly from GitHub
pip install git+https://github.com/YOUR_USERNAME/gwstrain.git
uv add "gwstrain @ git+https://github.com/YOUR_USERNAME/gwstrain.git"

# Editable developer install
git clone https://github.com/YOUR_USERNAME/gwstrain
cd gwstrain && uv sync --dev
```

### Publishing to PyPI
```bash
uv build            # creates dist/gwstrain-0.1.0-py3-none-any.whl
uv publish          # uploads to PyPI (requires API token)
```

---
## Summary: All 10 Guidelines Covered

| # | Guideline | Implementation in `gwstrain` |
|---|-----------|-----------------------------|
| 1 | **Plan Before You Code** | Problem statement + module map written before code |
| 2 | **Design for Modularity** | 4 independent modules: noise, waveform, io, matched_filter |
| 3 | **Write Clean & Readable Code** | Single-responsibility functions, descriptive names, NumPy docstrings |
| 4 | **Use Version Control** | `uv init` + `git init`, atomic commits, GitHub remote |
| 5 | **Test Regularly** | `pytest` unit + integration tests, CI runs on every push |
| 6 | **Peer Code Review** | PR template with guideline checklist |
| 7 | **Document Everything** | Docstrings, README quickstart, `pyproject.toml` metadata |
| 8 | **Strive for Reproducibility** | `seed` parameter everywhere, all params stored in HDF5 metadata |
| 9 | **Performance & Scalability** | FFT-domain matched filter, `cProfile` profiling cell |
| 10 | **Plan for Long-Term Maintenance** | GitHub Actions CI, `uv.lock` dependency pinning, PyPI publishing |